# Mini-projet — IA & gestion de stock

**Problème** : estimer une quantité à commander à partir du stock, des ventes, du nombre de jours de livraison (commande → réception) et de la saison.

**Données** : `../data/stock_synthetic.csv` (générées pour la démo — à documenter dans le rapport).

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path("..").resolve()
DATA = ROOT / "data" / "stock_synthetic.csv"
df = pd.read_csv(DATA)
df.head()

## Nettoyage

In [ ]:
df.info()
df.isna().sum()

In [ ]:
df = df.drop_duplicates()
df.describe()

## Visualisations

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
df["recommended_order_qty"].hist(bins=40, ax=ax[0], color="#2c7fb8", edgecolor="white")
ax[0].set_title("Distribution — quantité recommandée")
ax[0].set_xlabel("Unités")
df.boxplot(column="recommended_order_qty", by="category", ax=ax[1])
plt.suptitle("")
ax[1].set_title("Quantité par catégorie")
plt.tight_layout()
plt.show()

## Modèle — Random Forest (régression)

**Choix** : modèle non linéaire, robuste, interprétable via importances — adapté au sujet sans sur-ingénierie.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

X = df.drop(columns=["recommended_order_qty"])
y = df["recommended_order_qty"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

numeric = ["current_stock", "avg_weekly_sales", "lead_time_days", "month"]
categorical = ["category"]
prep = ColumnTransformer([
    ("num", "passthrough", numeric),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
])
pipe = Pipeline([
    ("prep", prep),
    ("model", RandomForestRegressor(n_estimators=120, max_depth=12, random_state=42, n_jobs=-1)),
])
pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)

mae = mean_absolute_error(y_test, pred)
rmse = mean_squared_error(y_test, pred) ** 0.5
r2 = r2_score(y_test, pred)
print(f"MAE: {mae:.2f} | RMSE: {rmse:.2f} | R²: {r2:.3f}")

In [ ]:
prep_fitted = pipe.named_steps["prep"]
model = pipe.named_steps["model"]
num_names = list(prep_fitted.transformers_[0][2])
ohe = prep_fitted.named_transformers_["cat"]
cat_names = list(ohe.get_feature_names_out(["category"]))
feat_names = num_names + cat_names
imp = model.feature_importances_
idx = imp.argsort()[::-1][:12]
plt.figure(figsize=(8, 5))
plt.barh([feat_names[i] for i in idx[::-1]], imp[idx[::-1]], color="#2c7fb8")
plt.xlabel("Importance")
plt.title("Variables les plus influentes")
plt.tight_layout()
plt.show()

## Synthèse

- Le modèle apprend une relation entre besoin (ventes × couverture), stock et saison.
- **Limites** : données synthétiques — à valider sur données réelles ; pas de contraintes métier (budget, palette) dans le modèle.
- **Intégration** : le même pipeline (`joblib`) est chargé par le backend Flask ; le formulaire HTML envoie les données en JSON à `POST /api/predict` (voir `frontend/`).